In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 2.2 Symmetry and Conservation: Noether's Theorem

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume II — Analytical Mechanics",
    number="2.2",
    title="Symmetry and Conservation: Noether's Theorem",
    blurb="Every continuous symmetry of the action hands us a conserved "
    "quantity — and break the symmetry and the conservation law dies "
    "with it. The deepest idea in classical mechanics, made concrete.",
    difficulty="intermediate",
    estimate="75–100 min",
)

## Notebook overview

In [§2.1](lagrangian-sympy.ipynb) we met a small miracle in passing: when the Lagrangian does not
depend on a coordinate (a *cyclic* coordinate), its conjugate momentum is
conserved. The cart's horizontal momentum was conserved because $\mathcal L$ did
not contain $x$; the orbital angular momentum was conserved because $\mathcal L$
did not contain the polar angle $\varphi$. Each time, a coordinate the Lagrangian
*ignored* corresponded to a quantity nature *conserved*.

**Noether's theorem** is the vast generalisation of that observation, and it is
the single most important structural idea in this volume. It says: *every
continuous symmetry of the action corresponds to a conserved quantity*, and it
gives an explicit formula that hands us the conserved quantity from the symmetry.
A cyclic coordinate is just its simplest face (the symmetry "shift $q_i$ by a
constant"). The general statement covers symmetries that mix several coordinates
at once (rotations, boosts, and subtler "hidden" symmetries) and it runs in
both directions: **no symmetry, no conservation law.** That converse is not a
footnote; it is something we will *demonstrate* by deliberately breaking a
symmetry and watching the corresponding conservation law die.

Computationally this notebook does three things. First, it tests symmetries
*symbolically*: invariance of $\mathcal L$ under a one-parameter transformation
means $\mathrm d\mathcal L/\mathrm d\varepsilon = 0$, which SymPy checks exactly.
Second, it *builds the conserved charge from the symmetry* with Noether's formula:
the theorem producing angular momentum, not us assuming it. Third, it confirms
the resulting conservation laws *numerically* along integrated trajectories, and
stress-tests the converse by breaking a symmetry on purpose. We reuse the
`euler_lagrange` engine from [§2.1](lagrangian-sympy.ipynb) throughout.

> **How to read the checks.** Each exercise ends with a validation that
> compares our result to an expected physical fact. A ✗ does *not* by itself
> mean the answer is wrong: it means the output didn't match what the check
> expected, which may be a real error, a different-but-valid convention (a
> sign, a unit, an array order), or simply too tight a tolerance. Treat a ✗ as
> a prompt to locate the discrepancy; passing is strong evidence of
> correctness, not proof.

> **Scope.** This is a working review, not a textbook chapter. For the full
> treatment (the variation of the action under a symmetry, the boundary-term
> subtleties, and the field-theory generalisation), see Nolting, *Theoretical
> Physics 2* {cite}`nolting2`, and Goldstein, Poole & Safko, *Classical
> Mechanics* {cite}`goldstein`.

## Theory in brief

### The statement

Consider a continuous one-parameter transformation of the coordinates,
$q_i \to q_i + \varepsilon\,\delta q_i$, where $\varepsilon$ is a small real
number and the $\delta q_i$ are the *generators* of the transformation. Suppose
this transformation leaves the action invariant: equivalently, it changes the
Lagrangian by at most a total time derivative, $\delta\mathcal L =
\varepsilon\,\mathrm df/\mathrm dt$ for some function $f$ (a change that cannot
affect the equations of motion). Then, *along any solution of the
Euler–Lagrange equations*, the **Noether charge**

```{math}
:label: eq-noether-charge
Q = \sum_i \frac{\partial \mathcal L}{\partial \dot q_i}\,\delta q_i - f
```

is conserved: $\mathrm dQ/\mathrm dt = 0$. The one-line derivation: the symmetry
gives $\mathrm d\mathcal L/\mathrm d\varepsilon = \sum_i(\partial_{q_i}\mathcal L
\,\delta q_i + \partial_{\dot q_i}\mathcal L\,\delta\dot q_i) = \dot f$; on a
solution the Euler–Lagrange equation {eq}`eq-el` lets us replace
$\partial_{q_i}\mathcal L$ by $\tfrac{\mathrm d}{\mathrm dt}\partial_{\dot
q_i}\mathcal L$, and the sum collapses to $\tfrac{\mathrm d}{\mathrm dt}\sum_i
\partial_{\dot q_i}\mathcal L\,\delta q_i = \dot f$: i.e. $\dot Q = 0$. The full
treatment is deferred to {cite}`nolting2,goldstein`.

### The three canonical cases

Each is {eq}`eq-noether-charge` evaluated for a particular symmetry.

- **Time-translation invariance.** If $\mathcal L$ has no explicit time
  dependence, shifting $t \to t + \varepsilon$ is a symmetry, and the conserved
  charge is the **energy**

```{math}
:label: eq-energy-cons
E = \sum_i \dot q_i\,\frac{\partial \mathcal L}{\partial \dot q_i} - \mathcal L .
```

- **Spatial-translation invariance.** If shifting every particle by the same
  displacement ($\delta x_i = \text{const}$) leaves $\mathcal L$ unchanged, the
  conserved charge is the total **linear momentum**

```{math}
:label: eq-momentum-cons
P = \sum_i \frac{\partial \mathcal L}{\partial \dot x_i} .
```

- **Rotational invariance.** If rotating the system leaves $\mathcal L$
  unchanged, the generator in 2-D is $(\delta x, \delta y) = (-y, x)$ and the
  conserved charge is the **angular momentum**

```{math}
:label: eq-angmom-cons
L_z = \sum_i \left(x_i\,\frac{\partial \mathcal L}{\partial \dot y_i}
      - y_i\,\frac{\partial \mathcal L}{\partial \dot x_i}\right) .
```

### Cyclic coordinates as the simplest case

The cyclic coordinates of [§2.1](lagrangian-sympy.ipynb) are Noether's theorem in miniature. If $\mathcal L$
does not depend on $q_i$, then the shift $\delta q_i = \text{const}$ (and all
other $\delta q_j = 0$) is a symmetry with $f = 0$, and {eq}`eq-noether-charge`
reduces to $Q = \partial\mathcal L/\partial\dot q_i = p_i$: the conjugate
momentum. So "cyclic coordinate $\Rightarrow$ conserved momentum" is the special
case where the symmetry generator is a single constant shift.

### The converse, and why it matters

The theorem is an *equivalence*, and the contrapositive carries the physics:
**break the symmetry and the conserved quantity stops being conserved.** This is
not a technicality. It is the reason a planet in a pure central field keeps its
angular momentum forever, while the same planet in a field with a preferred
direction does not. Exercise 5 makes this concrete: we add one
symmetry-breaking term to a potential and watch angular momentum come apart while
energy (protected by a symmetry we did *not* break) stays rock-steady.

> **Forward links.** The same idea, applied to *fields*, gives charge
> conservation from the gauge symmetry of electromagnetism (Volume III) and
> underlies the conservation laws of quantum mechanics (Volume VI). Noether's
> theorem is the thread that runs through all of them.

---
## Setup

We work with two flavours of SymPy object, and the distinction matters:

- **Algebraic symbols** ($x, y, v_x, v_y, \dots$) for the *symmetry* tests. Testing
  invariance and building the Noether charge {eq}`eq-noether-charge` are algebraic
  operations on $\mathcal L(q,\dot q)$ (no time integration is involved), so plain
  symbols are cleanest. Velocities are independent symbols here.
- **Functions of $t$** ($x(t), \dots$) for *deriving and integrating* equations of
  motion, exactly as in [§2.1](lagrangian-sympy.ipynb).

Three reusable helpers:

- `euler_lagrange(L, coords, t)`: the engine of [§2.1](lagrangian-sympy.ipynb), now imported from
  `ecp.mechanics` (promoted there once Volume II was complete). `invariance_defect`
  and `noether_charge` below stay local for now: used only here so far, they would
  be promoted when a later notebook reuses them.
- `invariance_defect(L, flow, eps)`: returns $\mathrm d\mathcal L/\mathrm
  d\varepsilon\big|_0$ under a one-parameter substitution `flow`; **zero means the
  transformation is a symmetry**.
- `noether_charge(L, vels, dq, f)`: assembles the conserved charge
  {eq}`eq-noether-charge` from the velocity symbols, the generator components
  $\delta q_i$, and the optional boundary function $f$.

In [ ]:
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import validate
from ecp.mechanics import euler_lagrange
from ecp.animate import show

t = sp.symbols("t")  # the time variable shared by all dynamical coordinates


def to_numeric(acc_dict, coords, t, param_values):
    """Lambdify Euler-Lagrange accelerations into a `solve_ivp` RHS (the 2.1 bridge).

    Compiles the symbolic accelerations, with parameters substituted, to a numeric ODE function.

    Parameters
    ----------
    acc_dict : dict
        The acceleration map.
    coords : sequence
        Coordinates.
    t : sympy.Symbol
        Time symbol.
    param_values : dict
        Parameter values.

    Returns
    -------
    callable
        ``rhs(t, y)`` for the integrator.
    """
    param_syms = list(param_values)
    pvals = [param_values[s] for s in param_syms]
    qdots = [sp.diff(q, t) for q in coords]
    args = list(coords) + qdots + param_syms
    acc_funcs = [sp.lambdify(args, acc_dict[sp.diff(q, t, 2)], "numpy") for q in coords]

    def rhs(_t, y):
        q = y[0::2]
        v = y[1::2]
        call = (*q, *v, *pvals)
        dy = np.empty_like(y)
        dy[0::2] = v
        dy[1::2] = [f(*call) for f in acc_funcs]
        return dy

    return rhs


def invariance_defect(L, flow, eps):
    """The derivative dL/dε at ε = 0 under a one-parameter flow.

    Zero means the transformation is a symmetry of the Lagrangian; this is the
    symbolic test of invariance.

    Parameters
    ----------
    L : sympy.Expr
        The Lagrangian.
    flow : callable
        The one-parameter coordinate substitution.
    eps : sympy.Symbol
        The group parameter.

    Returns
    -------
    sympy.Expr
        The defect (zero for a symmetry).
    """
    return sp.simplify(sp.diff(L.subs(flow, simultaneous=True), eps).subs(eps, 0))


def noether_charge(L, vels, dq, f=0):
    """The conserved Noether charge of a symmetry.

    Q = Σ_i (∂L/∂v_i)·δq_i − f (eq-noether-charge in the theory section), the
    constant of motion the symmetry guarantees.

    Parameters
    ----------
    L : sympy.Expr
        The Lagrangian.
    vels : sequence
        Velocity symbols.
    dq : sequence
        The infinitesimal coordinate changes.
    f : sympy.Expr, optional
        Boundary term (default 0).

    Returns
    -------
    sympy.Expr
        The conserved charge $Q$.
    """
    return sp.simplify(sum(sp.diff(L, v) * d for v, d in zip(vels, dq)) - f)

## Exercise 1 — Test a symmetry symbolically

Invariance is a statement one can *check*. For a one-parameter transformation
$q_i \to q_i(\varepsilon)$, the Lagrangian is a symmetry exactly when
$\mathrm d\mathcal L/\mathrm d\varepsilon\big|_{\varepsilon=0} = 0$ (possibly up to
a total time derivative $\mathrm df/\mathrm dt$). The `invariance_defect` helper
computes that derivative; a simplified result of zero is a proof of invariance.

The natural first test is the 2-D **central potential**, with Lagrangian
{eq}`eq-lagrangian` $\mathcal L = \tfrac12 m(v_x^2 + v_y^2) - V(r)$ and
$r=\sqrt{x^2+y^2}$ (we use $V(r) = -k/r$). Rotating the plane by an angle
$\varepsilon$ sends $(x,y)$ and $(v_x,v_y)$ through the same rotation matrix.
Because the rotation preserves both $x^2+y^2$ and $v_x^2+v_y^2$, the Lagrangian
should be untouched.

1. Build $\mathcal L$ in the algebraic symbols $x, y, v_x, v_y$
   (`sympy.symbols`).
2. Define the rotation flow on both positions and velocities and use
   `invariance_defect` to confirm
   $\mathrm d\mathcal L/\mathrm d\varepsilon|_0 = 0$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    sp.simplify(dL_deps_at_0) == 0,
    "the central-potential Lagrangian is rotationally invariant",
    f"dL/dε|0 = {dL_deps_at_0}",
)

## Exercise 2 — Build the Noether charge from the symmetry

This is the theorem doing its job: the symmetry *produces* the conserved
quantity. The infinitesimal rotation generator is read off from the flow by
differentiating at $\varepsilon=0$: $(\delta x, \delta y) =
\tfrac{\mathrm d}{\mathrm d\varepsilon}(x(\varepsilon), y(\varepsilon))|_0 =
(-y, x)$. The rotation leaves $\mathcal L$ strictly invariant, so $f=0$, and
Noether's charge {eq}`eq-noether-charge` is $Q = p_x\,\delta x + p_y\,\delta y$
with $p_x = \partial\mathcal L/\partial v_x$, $p_y = \partial\mathcal L/\partial
v_y$.

**Your task.** Take unit mass ($m=1$) for the algebra, build $Q$ with
`noether_charge`, and show it simplifies to $x\,v_y - y\,v_x$: the **angular
momentum** $L_z$ {eq}`eq-angmom-cons`. We did not assume angular momentum is
conserved; the rotational symmetry handed it to us.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    sp.simplify(Q_rot - (x * vy - y * vx)) == 0,
    "the Noether charge of rotational symmetry IS angular momentum",
    f"constructed charge Q = {Q_rot}",
)

## Exercise 3 — Rotational symmetry ⇒ angular momentum (numeric)

Symbol-level invariance promises conservation *along solutions of the equations
of motion*. Time to integrate an actual trajectory and watch $L_z$
{eq}`eq-angmom-cons` hold. We reuse the pipeline of [§2.1](lagrangian-sympy.ipynb): build the central-potential
Lagrangian in Cartesian coordinates $x(t), y(t)$ (Cartesian avoids the polar
coordinate singularity at the origin), let `euler_lagrange` derive the EOM, and
`to_numeric` + `solve_ivp` integrate a bound orbit.

**Your task.** Integrate the orbit with `to_numeric` + `scipy.integrate.solve_ivp`
(`DOP853`, `rtol=1e-11`, `atol=1e-12`) on a dense `t_eval`, compute
$L_z(t) = m(x\dot y - y\dot x)$ along it, confirm it is conserved, and plot the orbit.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(
    Lz_series,
    "angular momentum conserved under rotational symmetry",
    rel_drift=1e-6,
)

## Exercise 4 — Time-translation ⇒ energy (numeric)

The same trajectory carries a second conservation law. The central-potential
Lagrangian has no explicit time dependence, so time-translation $t \to t +
\varepsilon$ is a symmetry and the conserved charge {eq}`eq-energy-cons` is the
total energy $E = T + V = \tfrac12 m(\dot x^2 + \dot y^2) - k/r$. (For a Lagrangian
of the form $T - V$ with velocity-independent $V$, the abstract charge $\sum_i
\dot q_i\,\partial_{\dot q_i}\mathcal L - \mathcal L$ reduces to exactly $T + V$.)

**Your task.** Reconstruct $E(t)$ along the Exercise 3 trajectory
(`numpy.hypot` for $r$) and confirm it is conserved to tight tolerance: a
second, independent symmetry giving a second conserved quantity on the very
same orbit.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(
    E_series,
    "energy conserved under time-translation symmetry",
    rel_drift=1e-6,
)

## Exercise 5 — The contrast: break the symmetry, lose the law

This is the centrepiece, and it is where the converse earns its keep. We add a
single term to the potential, $V \to V + b\,x$: a uniform force in the
$x$-direction, as if the orbiting particle felt a constant pull. This term has a
*preferred direction*, so it breaks rotational invariance. But it carries no
explicit time dependence, so time-translation invariance survives untouched.

Noether's theorem then makes a sharp, falsifiable prediction: **angular momentum
should stop being conserved, while energy should remain conserved.** We check it
on both fronts: symbolically (the rotational defect is now nonzero) and
numerically (integrate the perturbed orbit and watch $L_z$ come apart while $E$
stays flat).

1. Recompute the rotational `invariance_defect` for $\mathcal L_{\text{broken}}
   = \tfrac12 m(v_x^2+v_y^2) + k/r - b\,x$ and confirm it is now **nonzero**.
2. Integrate the perturbed orbit (same `solve_ivp`/`DOP853` settings as
   Exercise 3); compute $L_z(t)$ and $E(t)$.
3. Make the figure that *is* the lesson: $L_z(t)$ for the symmetric orbit
   (flat) against $L_z(t)$ for the broken orbit (wildly varying) on one axes,
   and $E(t)$ relative drift for both (flat in both) below it.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    np.ptp(Lz_broken) / abs(Lz_broken[0]) > 1.0,
    "breaking rotational symmetry destroys angular-momentum conservation",
    f"L_z varies by {np.ptp(Lz_broken) / abs(Lz_broken[0]):.1f}× its initial value",
)
validate.conserved(
    E_broken,
    "energy is still conserved (time-translation symmetry intact)",
    rel_drift=1e-6,
)

## Exercise 6 — Translational symmetry ⇒ total momentum (worked animation)

Now a symmetry that mixes *particles* rather than directions. Two bodies interact
only through a potential that depends on their **separation**, $V(|\mathbf r_1 -
\mathbf r_2|)$, with no external field: here the gravitational
$V = -G m_1 m_2/|\mathbf r_1 - \mathbf r_2|$. Shifting *both* particles by the
same displacement leaves $|\mathbf r_1 - \mathbf r_2|$ (and hence $\mathcal L$)
unchanged: the system is **translationally invariant**. By
{eq}`eq-momentum-cons` the total linear momentum $\mathbf P = m_1\dot{\mathbf r}_1
+ m_2\dot{\mathbf r}_2$ is conserved, which means the centre of mass moves in a
straight line at constant velocity *no matter how complicated the internal orbit*.

This is the worked animation; you build the second in Exercise 7. We give the
pair a net drift velocity so the conserved momentum is plainly nonzero: the two
bodies wheel around each other while their centre of mass (marked) sails through
the frame on a perfectly straight track.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(
    P_total_mag,
    "total linear momentum conserved (translational symmetry)",
    rel_drift=1e-6,
)

## Exercise 7 — A hidden symmetry and its conserved charge (student-implemented animation)

Some symmetries are not obvious from looking at the Lagrangian. The **2-D
isotropic harmonic oscillator**, $\mathcal L = \tfrac12 m(\dot x^2 + \dot y^2) -
\tfrac12 m\omega^2(x^2 + y^2)$, is the classic example. It is plainly rotationally
invariant, so $L_z$ {eq}`eq-angmom-cons` is conserved. But it has *more* symmetry
than that: an extra "hidden" invariance whose Noether charge is a conserved
**symmetric tensor** $A_{ij} = \tfrac12(v_i v_j + \omega^2 x_i x_j)$ (the Fradkin
tensor). Its physical meaning is striking: the conservation of $A_{ij}$ is
exactly what forces the orbit to be a *closed, non-precessing ellipse*. The
principal axis of that ellipse points in a fixed direction for all time; that
fixed direction is the hidden conserved quantity made visible.

1. Build the isotropic-oscillator Lagrangian, integrate an orbit with
   `to_numeric` + `scipy.integrate.solve_ivp` (`DOP853`), and confirm $L_z$ is
   conserved (rotational symmetry, as in Exercise 3).
2. Form the Fradkin tensor components $A_{xx}, A_{yy}, A_{xy}$ along the
   trajectory and the principal-axis angle $\theta(t) = \tfrac12
   \operatorname{atan2}(2A_{xy},\, A_{xx}-A_{yy})$ (`numpy.arctan2`).
   Verify the orbit does **not** precess: $\theta(t)$ is constant.
3. **Build the animation** of the closed elliptical orbit, with the fixed
   conserved-axis direction drawn as a straight line through the centre. You
   have the trajectory `(x, y)` and the (constant) axis angle below; assemble
   a `FuncAnimation` showing the moving particle, its trail, and the fixed
   axis line, `plt.close(fig)`, then display with `ecp.animate.show`.

> A ✗ on the final check is about the **axis-angle series we computed** from the
> trajectory and the Fradkin tensor, not the animation. Any correct drawing of
> the same orbit is fine. If it fails, inspect $\theta(t) = \tfrac12
> \operatorname{atan2}(2A_{xy}, A_{xx}-A_{yy})$ along the solution.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.ptp(axis_angle),
    0.0,
    "the isotropic oscillator's orbit does not precess (hidden symmetry ⇒ conserved orientation)",
    atol=1e-2,
)

## Notebook summary

- **Noether's theorem** made operational: a symbolic test that a transformation leaves the
  Lagrangian invariant, and the construction of the conserved Noether charge from it.
- The canonical pairings confirmed numerically (conserved along `solve_ivp` trajectories):
  rotational symmetry ⇒ angular momentum, time-translation ⇒ energy, spatial translation ⇒
  total momentum; and the contrast that breaking the symmetry destroys the conservation law.

## Outlook

- **The energy/Jacobi distinction.** When $\mathcal L$ *is* explicitly
  time-dependent, energy {eq}`eq-energy-cons` is no longer conserved, but for a
  system written in a steadily rotating frame the **Jacobi integral** $h = \sum_i
  \dot q_i\,\partial_{\dot q_i}\mathcal L - \mathcal L$ still is. It is the charge
  of a *different* time-translation, and the gap between $h$ and $T+V$ is exactly
  the rotating-frame energy bookkeeping.
- **Scaling symmetry and the virial theorem.** Mechanical similarity (rescaling
  lengths and times together) is a symmetry of power-law potentials, and it
  yields the virial relation between average kinetic and potential energies
  (Landau & Lifshitz §10).
- **Why *continuous*?** Noether's theorem needs a continuous symmetry: the charge
  is built from the generator $\delta q_i$, which only exists for transformations
  one can do infinitesimally. Discrete symmetries (parity, time reversal) give
  powerful selection rules but no conserved Noether charge.
- **Noether in field theory.** Replace "coordinate" by "field" and the very same
  construction gives conserved *currents*. In electromagnetism the relevant
  continuous symmetry is **gauge invariance**, and the conserved charge is
  electric charge itself: a thread we pick up in Volume III.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()